# ControlPlane — TriviaQA extractionThe **only** stage that needs a GPU. Everything downstream runs from the cachesthis writes, which is what makes `/validate` fast enough to be a button a judgecan press.This session extracts **both** envelopes:| envelope | items | why ||---|---|---|| `triviaqa-600` | 2,400 questions → 1,200 train / 600 validation / 600 test | the tier ladder anchor || `triviaqa-longctx-600` | the 600 test questions, padded to 4–16k tokens | Beat 4's envelope shift |Both in one run. A two-session plan is a plan where the second session does nothappen, and Beat 4 has no measured basis without the long-context pass.**Runtime:** roughly 1–1.5 h for the short pass, 45–60 min for long context on aT4. Well inside a Kaggle session, with room for a restart.### Before you start- Accelerator: **GPU T4 ×2** (or P100). Internet: **on**, for the model and dataset.- Nothing here writes outside `/kaggle/working`.

## 0 — Pre-flightStops here if the environment cannot do the job.

In [ ]:
import subprocess, sysprint(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
%pip install -q bitsandbytes accelerate datasets

In [ ]:
import torchassert torch.cuda.is_available(), (    "No GPU. Extraction needs one; every other stage in this project does not.")free, total = torch.cuda.mem_get_info()print(f"{torch.cuda.get_device_name(0)}  {free/2**30:.1f} GiB free of {total/2**30:.1f} GiB")assert total / 2**30 > 14, (    f"{total/2**30:.1f} GiB total. Qwen2.5-7B in NF4 needs ~6 GiB for weights and "    "the long-context pass needs headroom for a 16k-token KV cache.")

## 1 — Get the repositoryClone rather than upload, so the notebook is running the same code that iscommitted and the artifacts record a commit that exists.

In [ ]:
# If you have pushed the repo, clone it. Otherwise upload it as a Kaggle Dataset# and point PROJECT at the mounted path.import os, sysfrom pathlib import PathPROJECT = Path("/kaggle/working/controlplane")if not PROJECT.exists():    # Replace with your repository URL, or attach the repo as a Dataset.    !git clone --depth 1 https://github.com/YOUR_USER/YOUR_REPO.git /kaggle/working/controlplanePROJECT = PROJECT / "round 2" if (PROJECT / "round 2").exists() else PROJECTsys.path.insert(0, str(PROJECT))os.chdir(PROJECT)print(PROJECT)

In [ ]:
!git -C "$(git -C . rev-parse --show-toplevel)" log --oneline -1

## 2 — Config and the padding assertionThe padding side is asserted at load, again before every batched forward pass,and a third time at validation by the fault-injection control. With rightpadding, position −1 of a batched sequence is a pad token, every activation isread from nothing, and the probe lands near 0.5 AUROC — which reads as *"theidea does not work"* rather than as a bug.

In [ ]:
import loggingfrom src.config import load_config, set_seeds, setup_loggingsetup_logging(logging.INFO)config = load_config("config.yaml")set_seeds(config.seed)print("config hash:", config.config_hash)print("model:", config.model.name, "| quantization:", config.model.quantization)print("aggregations:", list(config.probe.aggregations))

In [ ]:
from src.extract.model import load_modelloaded = load_model(config.model.name, quantization=config.model.quantization)print(loaded.provenance())print("layers resolved from fractional depths:", config.resolve_layers(loaded.num_hidden_layers))

## 3 — ExtractOne call. It deduplicates TriviaQA, splits **by question**, captures the paddingevidence, generates greedily, labels by alias match with the short-alias guard,and pools activations for every configured aggregation from the same forwardpass.Set `--smoke` equivalent by passing `n_questions=120` first if you want to checkthe wiring before spending the hour.

In [ ]:
from src.extract.pipeline import extract_triviaqaresult = extract_triviaqa(    config,    loaded,    n_questions=2400,      # 120 for a smoke run    batch_size=8,    long_batch_size=1,     # a 16k-token sequence does not batch on 16 GiB    max_new_tokens=32,    long_context=True,)result.report

### What to check before going further- **base rate** should be somewhere near 0.15–0.35. Exactly 0 or 1 raises, but a  base rate of 0.02 means almost every answer was judged correct and the alias  matching is probably too loose.- **match_rules** shows how labels were reached. A large `exact token match on  short alias` count is expected; a large `empty generation` count is not.- **token_length_max** for the long pass should land inside the configured  4,000–16,000 band.

In [ ]:
print("short :", len(result.short_evalset), "items,",      f"base rate {result.short_evalset.base_rate:.4f}", result.short_evalset.envelope_id)if result.long_evalset is not None:    print("long  :", len(result.long_evalset), "items,",          f"base rate {result.long_evalset.base_rate:.4f}", result.long_evalset.envelope_id)    print("long token lengths:", result.long_cache.token_lengths.min(),          "to", result.long_cache.token_lengths.max())

## 4 — Freeze and saveThe eval sets are content-hashed; the hash **is** the envelope id and thereforethe third element of every warrant key measured on them. The caches are largeand gitignored — download them, do not commit them.

In [ ]:
from src.evalsets import save_evalsetsave_evalset(result.short_evalset, "evalsets")short_path = result.short_cache.save("results/cache-triviaqa-600.npz")paths = [short_path]if result.long_evalset is not None:    save_evalset(result.long_evalset, "evalsets")    paths.append(result.long_cache.save("results/cache-triviaqa-longctx-600.npz"))for path in paths:    print(path, f"{path.stat().st_size / 2**20:.1f} MiB")

## 5 — Self-check, here rather than after downloadingTwo checks, both cheap and both worth failing on the GPU rather than on a laptopthree hours later.**Shape compatibility** asserts the measured path and the fixture path producethe same metric *structure* — same metrics present, same kinds, same units,intervals on both sides. Values must differ; shape must not. A mismatch meansthe two are not measuring the same thing and every fixture-versus-measuredcomparison in the repo is void.**The transfer** scores the short-context probe on long-context inputs withoutrefitting. That is the drift question — *what is **this** probe worth here?* —and it is what Beat 4 shows.

In [ ]:
from src.validation.runner import validatevariant = f"T1-{config.probe.aggregations[0]}"source = validate(    config, result.short_evalset, result.short_cache,    variant=variant,    detector_id=f"probe-{variant}",    detector_version=f"0.1.0+{loaded.name.split('/')[-1]}",    target_flag_rate=0.05,)print(source.summary())

In [ ]:
from src.validation.metrics_builder import assert_metric_shape_compatible, build_warrant_metricsfrom src.validation.synthetic import synthetic_cache, synthetic_evalsetfixture_evalset = synthetic_evalset(    eval_set_id="shape-check-synthetic",    n_items=len(result.short_evalset),    base_rate=result.short_evalset.base_rate,    seed=config.seed, items_per_question=1, declare_splits=True,)fixture = synthetic_cache(    fixture_evalset, seed=config.seed,    window=config.probe.rolling_window, stride=config.probe.rolling_stride,)fixture_metrics = build_warrant_metrics(    config, fixture.labels, fixture.matrix(variant)[:, 0], 0.5,    groups=fixture.question_ids,)assert_metric_shape_compatible(    fixture_metrics, source.metrics,    first_name="fixture path", second_name="measured extraction",)print("shape check passed — both paths produce the same metric structure")

In [ ]:
from src.detectors.probe import LinearProbefrom src.validation.evalsets import TRAIN, split_by_questionfrom src.validation.runner import validate_transferredif result.long_evalset is not None:    splits = split_by_question(result.short_evalset, seed=config.seed)    probe = LinearProbe(        source.probe_fit.C,        class_weight=config.probe.class_weight,        standardize=config.probe.standardize,        seed=config.seed,    ).fit(result.short_cache.matrix(variant), result.short_cache.labels, splits[TRAIN])    transferred = validate_transferred(        config, result.long_evalset, result.long_cache,        source=source, probe=probe, variant=variant,    )    print(transferred.summary())

## 6 — What to downloadTake all four. The caches are the expensive part and everything else in theproject runs from them on a laptop.```results/cache-triviaqa-600.npzresults/cache-triviaqa-longctx-600.npzevalsets/triviaqa-600.jsonevalsets/triviaqa-longctx-600.jsonresults/extraction.json```Then, locally:```bashpython scripts/03_matrix.py --config config.yaml```The `Outstanding measurement` section of `RESULTS.md` removes itself once thesetwo envelopes appear, and the matrix cells stop reading `FIXTURE — NOT MEASURED`.

In [ ]:
import shutilshutil.make_archive("/kaggle/working/controlplane-extraction", "zip", ".",                    base_dir=None, root_dir=".",                    )print("bundle written to /kaggle/working/controlplane-extraction.zip")